# E1 v1 — Safe resume after the pilot resources.json error

**Purpose:** Resume the *same* E1 full evaluation, preserving the original nine pilot batches, all existing E1 checkpoints, and frozen Q12 parent runs. No pilot replay. No new calibration. The original `p1_e1.py` is neither edited nor replaced.

### Why the first full-mode attempt failed
The first full-mode run successfully reused all 1,152 pilot executions, but the original coordinator tried to rewrite the immutable `pilot/resources.json` with a newly measured `real_wall_s`. That timing value necessarily changed, so the immutable comparison rejected the file.

This recovery launcher checks **every completed batch, the raw/compact file hashes, the manifest, and all stable resource totals**, then returns the original resource summary unchanged. Incomplete stages use the unchanged E1 evaluator and its original checkpoint/resume behavior. It also handles a future resume after the full evaluation has already completed.

**Use the exact run directory from your pilot; do not create a new one.** The provided path is filled in from your report. Run All once, after checking CPU and available RAM. For troubleshooting, the original files remain on the server.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.home() / "P1"
E1_RUN_DIR = PROJECT_DIR / "results" / "e1-20261025T040643708320Z"
Q12_RUN_DIR = PROJECT_DIR / "results" / "q12-20261013T034634906569Z"
Q12_CODE_DIR = None  # Auto-locate pinned Q12 code.

MODE = "full"
WORKERS = 8
PYTHON = None  # Auto-detect pinned interpreter.
REPO = None    # Auto-detect pinned Qoala repository.
EXPORT_COMPACT = True


## Install the independent recovery launcher
The existing, pinned `p1_e1.py` and its experiment-source hash remain unchanged. The launcher is a separate file and is itself checked against a fixed SHA-256 value.

In [ ]:
import base64, gzip, hashlib, json, os
from pathlib import Path

ORIGINAL_BUNDLE_HASH = "b1a9477d54794bab92ed32a30e8f0736dd4ee791bc49e9cf817e960a87426034"
ORIGINAL_RUNNER_SHA256 = "e75bf81ee0298f39531cf40c4549af078b24b97c2f88207fd031516f3f444a74"
RESUME_RUNNER_SHA256 = "2b36024cd6b7375c7186e063b03b02851cd4e891cd019a0caf9791488605de57"
RESUME_EMBEDDED = "H4sIAAAAAAAC/61YW2/cuBV+n1/BerGQZjOWPY69dibVg5sONkbbjeF4gQUCQ+BI1AxjSdSSlC91/d97Di+6jS/ZtH4YSxT58Vy/c8idnZ0LppqSkeWc3MzJLdcb0Wgi2a3kmldrkoqyLphmGVGarhnRvMThkmmaUU2jyeRywxUpaFOlGyYJL2shtSJ6w4iQfM0rWszIhqrNbs2rCnDqecLmUX1PcAWt1iyLyKeK0Ml4K67JDS04bMMUoUXR4pEV1emGwH7pdS14pQ0+UzNS0ornTOnZhFYGZlUwUEaJRqYgu9C0gFkgWwWjupHVUNB2qoq+KlGR1T1sHU1OK8IrL50TTmkOEjWKjRA6oUAP2YDGMiK/CmJxJ0LiHk0BFqKSOTtrEEcLkvM7wAJjrpp1NNnZ2ZnkUpQkSfIGRGVJ4oxLaFWBJpqLSk38kFzXVCrm30uqN/5Z8TVIZsFqGC/4yiOd96YZvxCqCJtPJsvfz5cfLpd/T5bz5PPH04Ojn0lMAnZ8tMpP5oztH7w7yd++O3o7T/PD/fTw6PAdzfePT1YHh6t3x+lBfnJysH+cZ/tv50fzn/O3+eHhIT0+DCbJp4uzX85+Pf1nsvx9+eG3y2Xy+fL0lyWAQ0ywO5Y2miXGwpPJJGN5GwFZ0oaH/R6a3xmRQuiZM++M1FJokQoIuVoUPOVMTRcTAn84DXZBhUN8nppRHy8J2gU+m1l7zsV7JPDfTTgEFsiHyHNrhjFkF/2AIVBLdsNFo4p7cHXFIWJdpL8noNcoEE3kkEwMA+pWyOvIIPKcQBAMFQADcqVVOCUY/fh5KGz73ZrEaoNZANJVzAl62qZ8C05SWqH0KcsY8cTQQhOaaxCVklRCFkbkcuOhxtogjGRrBs/g0R4CKIMSiwpMA89giXtUHTUwIrcq/7/VfdmKkKzfhkq5YuSiqYAc2VJKIcPgg5CyqfUWf4YlV8owqNtzuiABeQNfpYnLPTNtOvXiQVrkvGBIcCE8f1xeLPcCT6HBlPwlJtu5+rJon7xbgPMdMyI8cWz8HhTOGyMjkJI05SFo5fEJ9iWwSxO1obBjcPU9ksBbCwgOh0iw3kB2x3RwFUNsCfyUPD7hBxKBmTK+BjOHIz54TqRznHZvDQKuMrLAbmaNpuoagoDJxBYgA4+DSbppqmtlKWnILHaSZDQLB1HWKlCwKhwBG6/ieBcjr8TaMMbavV2dFDC7VQbC7U1PTm/bxBsSzUcGhnMf7Hx2V7MUSbin4Jcr8ymHfOEzayUomIRVEDmY6Vv6derkosiAGSyL2tjfCwMza/fH/aMs+JFP28npNbtXMFdBuWJZ+IA5o78EKS1rCjUuuJoaGTRubrZ8nBFYEgN3diCtAl/FCrAynuqw/Yh/RojYVRcDE5vf2WBWDRHCVPwA+AsfWl/g5cqIAA8ohBH4cbjwj/lB4lIn4zJGHezrdHteKjKW2N4m7gIdvgRX5p8HslOCqyECm3cA8bN5O1wDTD2IBJgSPxki3bLOtJauTKxbv+4FYGRbBk1M922/GO77RFj/zUQvuinjec6kIqaFgZ9/s443bDxLt+O0k+YHclalRZMBq1j7EJGPiiy9JVrSFDtH5GmaItmnQmYKS0oPCesuhCkEhVYR+YAlGfs+2s3vSAsFxoKkJcd+EJMv6tJX3MJmBmfWb2BNwnXvTpvZwGIDQyM9IFjLFSZGp99qVSv2k9TwhCm3kj6idc2qLDTZw6uM3cWQ+T/91GnglIzNbw8qY4UxwriiD8LDD34n7Q3CpRNJjaiv6z86jh7W+hHlQYMOm405w/gZFG1K4Os6BF/MxnzXS20zAOn8FO13s2y8JWndJA4Z8hYtiYlvRsF/VQavhm5KJJu2WGzB3MLh6QkcO/xNQCW9SxyYpGVS8lUMQwO0mtHrRCqFH19BMz5onexppt/pDEulXdlWTMV05ycT/zgy9NKU/Ic8QCsOrGXVDB5fDqbfKg/QxsUuGKykErt1VmTKZ8coRKZt7QOy7yUsHFwaZsrgQLCIa1YOukfQqIW0FaSfCgbm9ay+8GdcL/NzOUDC4A1s8iaYjhuBlWLyBnZEe2FBbmUamPHKeyHEVo1DmwgYVcrCwfpZiBSQF4Lqqemjn524Er6z8H+u68YTbMQVnpT0aJFBHMr7V7L/SpN0VpmjpGO+ljSknTQyRi0NCQcPZuxxAatsR3zDJIdwyMZ9/YI8dAbbsZSwc7WYPTp6eE+CnpIAPGjvHm2FyvqOeo8nLqtiV7GMrijvGG5gjUU0zx+JCsABDZT+S9m0jGeOP62k7oytaM6SweF7eLiWrBYvHbFnjmjaszaYGAxluPK187vZwUG3yG2vPmkTxCNye0bAE9zWsa6d1Vf2mRuH/vZGv+dkmHndnLFKyiufvnXs712iU7mGbrfS5/gmQ+g6UslrvKOJg494+4VGdrds9kzVu2xzuWtvg6CRc0EY0SxLqAMOg91dEO4r0EIw0/c1i/EuA2T/o+Ggcs/L2+ugVdyFQP+uddhC9hY+NxVt+KfxQaZdaIL/9LrSyJRuBIcgjqH28EKAVYK8KQroaMFNtCl0bN+fA3F+dZsjX/l1J3YJTIa2OzL+xZXA2j4e8S1yAMA98wXgM8szLTApGzyCwSlWKK75Te/Q+j8e6tu9+id59NPoCI93k8JHlT9MY2WRHI3g5MGgxtslc18Bxyq8LmzKWQ6FnvU4FWT7fPnpPMJaOx0x7T/Y/UpQmZ15GFOQMMZ7wO+RM0u87cx8FwSyYToZdsWrWieRvbCM7L/QvX3GLL6c9QBfnny5vPjX9uzxLSOeJrfYz8+E8AzR05j64LUkqcAkSRLHQZKg3EkSLBwfTP4Ldmxr00oXAAA="

PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
RUN_DIR = Path(E1_RUN_DIR).expanduser().resolve()
E1_CODE_DIR = PROJECT_DIR / "experiment_code" / ("e1-" + ORIGINAL_BUNDLE_HASH[:16])
E1_RUNNER = E1_CODE_DIR / "p1_e1.py"
if not RUN_DIR.is_dir() or not (RUN_DIR/"pilot"/"resources.json").is_file():
    raise RuntimeError("Existing E1 pilot run not found. Verify E1_RUN_DIR; do not start a new run.")
if not E1_RUNNER.is_file() or hashlib.sha256(E1_RUNNER.read_bytes()).hexdigest() != ORIGINAL_RUNNER_SHA256:
    raise RuntimeError("Original E1 runner not found with its pinned hash: " + str(E1_RUNNER))
protocol=json.loads((RUN_DIR/"protocol.json").read_text())
if protocol["source_sha256"] != ORIGINAL_RUNNER_SHA256:
    raise RuntimeError("Existing E1 run protocol does not match the original runner")
if not (RUN_DIR/"PILOT_PASSED.json").is_file():
    raise RuntimeError("Pilot pass marker missing; stop and inspect the original run")
blob=gzip.decompress(base64.b64decode(RESUME_EMBEDDED))
assert hashlib.sha256(blob).hexdigest()==RESUME_RUNNER_SHA256
RESUME_RUNNER=E1_CODE_DIR / "p1_e1_resume.py"
if RESUME_RUNNER.exists():
    if RESUME_RUNNER.read_bytes()!=blob:
        raise RuntimeError("Existing recovery file differs; do not overwrite: "+str(RESUME_RUNNER))
else:
    temp=RESUME_RUNNER.with_name(RESUME_RUNNER.name+'.tmp-'+str(os.getpid()))
    temp.write_bytes(blob)
    os.replace(temp,RESUME_RUNNER)
print("Original E1 file: ",E1_RUNNER)
print("Original E1 hash: ",ORIGINAL_RUNNER_SHA256)
print("Recovery launcher:",RESUME_RUNNER)
print("Existing results:  ",RUN_DIR)
print("Recovered protocol:",protocol["version"])


## Preflight: pinned simulator, original offline tests, and free memory

In [ ]:
import os, sys, shutil, subprocess

assert MODE in ("pilot", "full")
assert isinstance(WORKERS, int) and not isinstance(WORKERS, bool) and WORKERS >= 1
available_cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if available_cpus and WORKERS > available_cpus:
    raise RuntimeError(f"WORKERS={WORKERS}, but Jupyter has only {available_cpus} available CPU cores")

RUN_ENV = os.environ.copy()
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                 "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    RUN_ENV[variable] = "1"
RUN_ENV["PYTHONUNBUFFERED"] = "1"
RUN_ENV.pop("PYTHONOPTIMIZE", None)

# Defend against another memory failure. The reported memory is the approximate
# available budget, NOT the installed physical RAM amount.
def estimated_available_mib():
    candidates=[]
    meminfo=Path("/proc/meminfo")
    if meminfo.exists():
        for line in meminfo.read_text().splitlines():
            if line.startswith("MemAvailable:"):
                candidates.append(int(line.split()[1]) / 1024)
                break
    cg_max=Path("/sys/fs/cgroup/memory.max")
    cg_used=Path("/sys/fs/cgroup/memory.current")
    if cg_max.exists() and cg_used.exists() and cg_max.read_text().strip() != "max":
        candidates.append(max(0,(int(cg_max.read_text())-int(cg_used.read_text()))/1048576))
    return min(candidates) if candidates else None

free_mib=estimated_available_mib()
required_mib=WORKERS*1600+2048
print("CPU cores visible:", available_cpus)
print("Estimated free RAM:", "unknown" if free_mib is None else f"{free_mib/1024:.1f} GiB")
print("E1 safety budget:", f"{required_mib/1024:.1f} GiB available for {WORKERS} workers")
if free_mib is not None and free_mib < required_mib:
    raise RuntimeError("Insufficient available RAM for requested workers. Reduce WORKERS "+
                       "or run when P1_Resume_After_MemoryError is not consuming memory.")

PARENT_SOURCE_HASH = "470078ab4b2c014d42b97dbba496934a0195ae13a58d3404274c06b8e5b3b7b4"
Q12_CODE_DIR = Path(Q12_CODE_DIR).expanduser().resolve() if Q12_CODE_DIR else PROJECT_DIR / "experiment_code" / ("q12-" + PARENT_SOURCE_HASH[:16])
Q12_RUN_DIR = Path(Q12_RUN_DIR).expanduser().resolve()
if not (Q12_CODE_DIR / "q12.py").is_file():
    raise RuntimeError("Original Q12 code not found: " + str(Q12_CODE_DIR))
if not (Q12_RUN_DIR / "COMPLETE.json").is_file():
    raise RuntimeError("Original complete Q12 run not found: " + str(Q12_RUN_DIR))
RUN_ENV["P1_Q12_CODE_DIR"] = str(Q12_CODE_DIR)

repo_candidates=([Path(REPO).expanduser()] if REPO else
                 [PROJECT_DIR/name for name in ("qoala-sim","p1-qoala","qoala")] +
                 [Path.home()/"p1-qoala",Path.home()/"qoala-sim"])
repos=[]
for candidate in repo_candidates:
    if not candidate.is_dir(): continue
    r=subprocess.run(["git","-C",str(candidate),"rev-parse","HEAD"],capture_output=True,text=True)
    if r.returncode==0 and r.stdout.strip()=="31fa2c9699578d227736557f35cd01a134c4bbde":
        repos.append(candidate.resolve())
if not repos: raise RuntimeError("Pinned Qoala checkout not found; set REPO in first code cell")
REPO=repos[0]

if PYTHON:
    python_candidates=[str(Path(PYTHON).expanduser())]
else:
    python_candidates=[str(PROJECT_DIR/".venv-p1"/"bin"/"python"),
                       str(PROJECT_DIR/".venv"/"bin"/"python"),sys.executable]
python_candidates=list(dict.fromkeys(python_candidates))
preflight="import json,sys; import q6_reference; print(json.dumps(q6_reference.check_repo(sys.argv[1])))"
errors=[];chosen=None
for candidate in python_candidates:
    if not Path(candidate).is_file():continue
    r=subprocess.run([candidate,"-c",preflight,str(REPO)],cwd=str(Q12_CODE_DIR),env=RUN_ENV,capture_output=True,text=True)
    if r.returncode==0:
        chosen=candidate
        print("Validated interpreter:",chosen)
        print("Pinned Qoala checkout:",REPO)
        print(r.stdout)
        break
    errors.append(candidate+"\n"+r.stderr[-1200:])
if chosen is None:
    print("\n".join(errors))
    raise RuntimeError("No valid Q12 NetSquid/Qoala interpreter found; specify PYTHON")
PYTHON=chosen
for script, cwd in ((Q12_CODE_DIR/"q12_tests.py",Q12_CODE_DIR),
                    (E1_CODE_DIR/"p1_e1_tests.py",E1_CODE_DIR)):
    r=subprocess.run([PYTHON,"-u",str(script)],cwd=str(cwd),env=RUN_ENV,capture_output=True,text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr)
        raise RuntimeError("Offline checks failed: "+str(script))
print("Preflight passed. No new NetSquid simulation has run yet.")


## Resume full E1 run — validation of completed pilot, then fresh evaluation
The recovery launcher reuses the existing pilot `resources.json` without rewriting it. Full execution uses eight workers (or the smaller number you set above) and is separately checkpointed.

In [ ]:
import os, signal, subprocess
from datetime import datetime, timezone

assert MODE == "full", 'Set MODE="full" to run the remaining evaluation'
launcher_log=RUN_DIR / ('recovery-launcher-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'.log')
cmd=[PYTHON,'-u',str(RESUME_RUNNER),'--project',str(PROJECT_DIR),
     '--q12-run',str(Q12_RUN_DIR),'--q12-code',str(Q12_CODE_DIR),
     '--repo',str(REPO),'--run-dir',str(RUN_DIR),'--mode',MODE,'--workers',str(WORKERS)]
print('E1 RUN DIRECTORY:',RUN_DIR)
print('Recovery log:',launcher_log)
print('Mode:',MODE,'| Workers:',WORKERS)
with launcher_log.open('w') as log:
    process=subprocess.Popen(cmd,cwd=str(E1_CODE_DIR),env=RUN_ENV,
                             stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1,start_new_session=True)
    try:
        for line in process.stdout:
            log.write(line);log.flush();print(line,end='',flush=True)
        rc=process.wait()
    except BaseException:
        for sig in (signal.SIGINT,signal.SIGTERM,signal.SIGKILL):
            if process.poll() is not None:break
            try:os.killpg(process.pid,sig)
            except ProcessLookupError:break
            try:process.wait(timeout=8);break
            except subprocess.TimeoutExpired:pass
        raise
    finally:process.stdout.close()
if rc:
    raise RuntimeError('E1 recovery failed (exit '+str(rc)+'). Check '+str(launcher_log))
print('E1 full evaluation complete. Original run directory preserved:',RUN_DIR)


## Inspect the results and create a compact review archive
The archive contains hashes, summaries, logs, original source and recovery launcher. Full simulator raw event traces remain in the original E1 run directory.

In [ ]:
import json, zipfile
from IPython.display import FileLink, display

for name in ('PILOT_PASSED.json','COMPLETE.json','analysis.json'):
    p=RUN_DIR/name
    if p.is_file():
        print(name,p.stat().st_size,'bytes')
        if name=='analysis.json':
            data=json.loads(p.read_text())
            for r in data['contrasts']:
                print(r['contrast'],'delta (pp):',round(r['campaign_mean_difference_pp'],4),
                      'CI:',[round(v,4) for v in r['interval_pp']],
                      'primary:',r['primary'])
archive=RUN_DIR/('E1_Review_'+('complete' if (RUN_DIR/'COMPLETE.json').exists() else 'partial')+'_safe_resume.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for p in sorted(RUN_DIR.rglob('*')):
        if p==archive or not p.is_file() or '.tmp-' in p.name:continue
        if p.name.endswith(('.json','.log')) or (EXPORT_COMPACT and p.name.endswith('.compact.jsonl.gz')):
            z.write(p,p.relative_to(RUN_DIR))
    for p in (E1_CODE_DIR/'p1_e1.py',E1_CODE_DIR/'p1_e1_tests.py',RESUME_RUNNER):
        z.write(p,'source/'+p.name)
print('Review archive:',archive)
print('Original raw event traces remain under:',RUN_DIR)
display(FileLink(str(archive)))
